In [1]:
import csv
import json
from pathlib import Path


def write_jsonl(path, rows):
    """Write a list of dictionaries to a file, one JSON object per line."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"wrote {len(rows)} rows -> {path}")


def read_csv(path):
    """Read a CSV file into a list of dictionaries, one per row."""
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [5]:
import pandas as pd

NAMES = ("admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion",
         "curiosity", "desire", "disappointment", "disapproval", "disgust", "embarrassment",
         "excitement", "fear", "gratitude", "grief", "joy", "love", "nervousness", "optimism",
         "pride", "realization", "relief", "remorse", "sadness", "surprise", "neutral")

In [13]:
EMOTIONS = ["no emotion", "anger", "disgust", "fear", "happiness", "sadness", "surprise"]
ACTS = {1: "inform", 2: "question", 3: "directive", 4: "commissive"}

for split in ("train", "validation", "test"):
    files = [f for f in (RAW / "dailydialog").glob("*.parquet") if split in f.name]
    df = pd.read_parquet(files[0])
    converted = []
    for r in df.to_dict("records"):
        for t, a, e in zip(r["dialog"], r["act"], r["emotion"]):
            if not t.strip() or int(a) not in ACTS:
                continue
            converted.append({"text": t.strip(), "y": {"dd_emotion": EMOTIONS[int(e)], "dd_act": ACTS[int(a)]}})
    write_jsonl(OUT / "dailydialog" / f"{split}.jsonl", converted)

wrote 87170 rows -> /home/vbhv/Projects/ECHO/research/datasets/prepared/dailydialog/train.jsonl
wrote 8069 rows -> /home/vbhv/Projects/ECHO/research/datasets/prepared/dailydialog/validation.jsonl
wrote 7740 rows -> /home/vbhv/Projects/ECHO/research/datasets/prepared/dailydialog/test.jsonl
